In [1]:
# %%
"""
SageMaker Pipeline for Hybrid Recommendation System
Flow: Train -> Deploy -> Test -> Cleanup
Bucket: my-ml-hybrid-recsys-bucket
"""

import sagemaker
from sagemaker import get_execution_role
from sagemaker.sklearn.estimator import SKLearn

role = get_execution_role()
sess = sagemaker.Session()
bucket = "my-ml-hybrid-recsys-bucket"
prefix = "recommender"

/home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker_core/main/shapes.py:2509: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  class MonitoringDatasetFormat(Base):


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
import torch
import sys

print(f"PyTorch Version: {torch.__version__}")
print(f"Python Version:  {sys.version.split()[0]}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

PyTorch Version: 2.4.0+cu121
Python Version:  3.10.19
CUDA Available:  False


In [3]:
# %%
# Define SKLearn estimator
from sagemaker.pytorch import PyTorch
estimator = PyTorch(
    entry_point="train.py",
    source_dir="src",
    role=role,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    framework_version="2.4.0",
    py_version="py311",
    output_path=f"s3://{bucket}/{prefix}/output",
    base_job_name="hybrid-recsys",
    sagemaker_session=sess
)

In [4]:
# Start training
estimator.fit({"training": f"s3://{bucket}/{prefix}/input-data/"})

[03/04/26 06:24:39] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=448847;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=247349;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#90\90]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[03/04/26 06:24:46] INFO     image_uri is not presented, retrieving image_uri based on            ]8;id=300223;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=601298;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/image_uris.py#675\675]8;;\
                             instance_type, framework etc.                                                         

                    INFO     Creating training-job with name: hybrid-recsys-2026-03-04-06-24-39-581 ]8;id=79672;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=209514;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\

2026-03-04 06:24:47 Starting - Starting the training job
2026-03-04 06:24:47 Pending - Training job waiting for capacity...
2026-03-04 06:25:08 Pending - Preparing the instances for training...
2026-03-04 06:25:34 Downloading - Downloading input data...
2026-03-04 06:26:00 Downloading - Downloading the training image........................
2026-03-04 06:30:17 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 550.127.08
Current installed NVIDIA driver version is 570.211.01
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-03-04 06:30:32,912 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-03-04 06:30:32,931 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-03-04 06:30:32,941 sage

In [6]:
# %%
# Deploy to real-time endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    entry_point="inference.py",
    source_dir="src"
)

print("Endpoint deployed: ", predictor.endpoint_name)

[03/04/26 06:39:04] INFO     Repacking model artifact                                                  ]8;id=621264;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/model.py\model.py]8;;\:]8;id=975160;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/model.py#819\819]8;;\
                             (s3://my-ml-hybrid-recsys-bucket/recommender/output/hybrid-recsys-2026-03             
                             -04-06-24-39-581/output/model.tar.gz), script artifact (src), and                     
                             dependencies ([]) into single tar.gz file located at                                  
                             s3://sagemaker-ap-southeast-2-869540999151/hybrid-recsys-2026-03-04-06-39             
                             -04-109/model.tar.gz. This may take some time depending on model size...              

[03/04/26 06:39:17] INFO     Creating model with name: hybrid-recsys-2026-03-04-06-39-04-109        ]8;id=40038;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=873598;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/session.py#4025\4025]8;;\

[03/04/26 06:39:18] INFO     Creating endpoint-config with name                                     ]8;id=455022;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=484032;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/session.py#5820\5820]8;;\
                             hybrid-recsys-2026-03-04-06-39-04-109                                                 

                    INFO     Creating endpoint with name hybrid-recsys-2026-03-04-06-39-04-109      ]8;id=278376;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=640064;file:///home/sagemaker-user/.conda/envs/hybrid_recsys_env/lib/python3.10/site-packages/sagemaker/session.py#4642\4642]8;;\

----------!Endpoint deployed:  hybrid-recsys-2026-03-04-06-39-04-109


In [7]:
# %%
# Test the endpoint
import json
import boto3

runtime = boto3.client("sagemaker-runtime")
payload = json.dumps({"userId": 42, "top_n": 10})

response = runtime.invoke_endpoint(
    EndpointName=predictor.endpoint_name,
    ContentType="application/json",
    Body=payload
)

result = json.loads(response["Body"].read().decode())
print("Recommendation result:")
print(json.dumps(result, indent=2))

Recommendation result:
[
  {
    "movieId": 16,
    "title": "Casino (1995)",
    "genres": "Crime|Drama"
  },
  {
    "movieId": 2706,
    "title": "American Pie (1999)",
    "genres": "Comedy|Romance"
  },
  {
    "movieId": 296,
    "title": "Pulp Fiction (1994)",
    "genres": "Comedy|Crime|Drama|Thriller"
  },
  {
    "movieId": 2028,
    "title": "Saving Private Ryan (1998)",
    "genres": "Action|Drama|War"
  },
  {
    "movieId": 2423,
    "title": "Christmas Vacation (National Lampoon's Christmas Vacation) (1989)",
    "genres": "Comedy"
  },
  {
    "movieId": 2502,
    "title": "Office Space (1999)",
    "genres": "Comedy|Crime"
  },
  {
    "movieId": 553,
    "title": "Tombstone (1993)",
    "genres": "Action|Drama|Western"
  },
  {
    "movieId": 223,
    "title": "Clerks (1994)",
    "genres": "Comedy"
  },
  {
    "movieId": 1784,
    "title": "As Good as It Gets (1997)",
    "genres": "Comedy|Drama|Romance"
  },
  {
    "movieId": 1222,
    "title": "Full Metal Jacket 

In [1]:
# %%
# Delete endpoint to avoid charges
predictor.delete_endpoint()
print("Endpoint deleted successfully. Project completed.")

NameError: name 'predictor' is not defined